# ✅ Solution 2: Deteksi Outlier dengan Mean-Median Gap

**Approach**: Hitung gap per hari, threshold-based anomaly detection  
**Key Insight**: Gap mean-median > 10% atau skewness > 1.0 menandakan ada outlier signifikan.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

data_senin = np.random.normal(120, 30, 50).clip(50, 250)
data_selasa = np.random.normal(110, 25, 55).clip(50, 200)
data_rabu = np.concatenate([np.random.normal(100, 20, 45), [800, 1200, 1500]])
data_kamis = np.random.normal(130, 35, 48).clip(50, 250)
data_jumat = np.concatenate([np.random.normal(115, 28, 52), [950, 1100, 2000, 3000]])
data_sabtu = np.random.normal(140, 30, 60).clip(60, 280)
data_minggu = np.random.normal(125, 32, 45).clip(50, 260)

penjualan = {
    'Senin': data_senin, 'Selasa': data_selasa, 'Rabu': data_rabu,
    'Kamis': data_kamis, 'Jumat': data_jumat, 'Sabtu': data_sabtu, 'Minggu': data_minggu,
}

## 📖 Penjelasan Pendekatan

1. Hitung mean, median, gap untuk setiap hari
2. Gunakan 2 indikator anomali:
   - `gap_persen > 10%` : mean tertarik > 10% dari median
   - `skewness > 1.0` : distribusi sangat miring
3. Hari yang memenuhi salah satu/kedua kriteria = anomali
4. Investigasi transaksi spesifik yang menyebabkan anomali

In [ ]:
# SOLUSI TODO 1: Statistik per hari
stats_list = []
for hari, data in penjualan.items():
    mean_val = np.mean(data)
    median_val = np.median(data)
    stats_list.append({
        'hari': hari,
        'n_transaksi': len(data),
        'mean': round(mean_val, 2),
        'median': round(median_val, 2),
        'gap': round(mean_val - median_val, 2),
        'gap_persen': round((mean_val - median_val) / median_val * 100, 2),
        'skewness': round(pd.Series(data).skew(), 4),
    })

df_stats = pd.DataFrame(stats_list)
print('Statistik Penjualan per Hari (dalam ribu Rupiah):')
display(df_stats)

In [ ]:
# SOLUSI TODO 2: Identifikasi hari anomali
df_stats['anomali'] = (df_stats['gap_persen'] > 10) | (df_stats['skewness'] > 1.0)

print('=== Hari dengan Anomali ===')
hari_anomali = df_stats[df_stats['anomali']]
display(hari_anomali)

print('\n=== Transaksi Mencurigakan (> mean + 2*std) ===')
for _, row in hari_anomali.iterrows():
    hari = row['hari']
    data = penjualan[hari]
    threshold = np.mean(data) + 2 * np.std(data)
    suspicious = data[data > threshold]
    print(f'  {hari}: threshold = {threshold:.0f}rb')
    print(f'    Transaksi di atas threshold: {suspicious}')
    print(f'    Jumlah: {len(suspicious)} transaksi mencurigakan')
    print()

In [ ]:
# SOLUSI TODO 3: Visualisasi boxplot per hari
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Boxplot
hari_order = ['Senin', 'Selasa', 'Rabu', 'Kamis', 'Jumat', 'Sabtu', 'Minggu']
data_for_box = [penjualan[h] for h in hari_order]
colors = ['#4CAF50' if h not in hari_anomali['hari'].values else '#F44336' for h in hari_order]

bp = axes[0].boxplot(data_for_box, labels=hari_order, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

overall_median = np.median(np.concatenate(list(penjualan.values())))
axes[0].axhline(overall_median, color='blue', linestyle=':', lw=1.5, label=f'Overall Median = {overall_median:.0f}')
axes[0].set_title('Boxplot Penjualan per Hari\n(Merah = Anomali)', fontweight='bold')
axes[0].set_ylabel('Nilai Transaksi (ribu Rp)')
axes[0].legend()

# Bar chart gap
bar_colors = ['#F44336' if a else '#4CAF50' for a in df_stats['anomali']]
axes[1].bar(df_stats['hari'], df_stats['gap_persen'], color=bar_colors, edgecolor='black', alpha=0.7)
axes[1].axhline(10, color='red', linestyle='--', lw=1.5, label='Threshold 10%')
axes[1].set_title('Mean-Median Gap (%) per Hari\n(Merah = Anomali)', fontweight='bold')
axes[1].set_ylabel('Gap (%)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 📌 Takeaways

- **Rabu dan Jumat** teridentifikasi anomali — ada transaksi besar yang tidak biasa
- Mean-median gap > 10% = indikator kuat ada outlier
- Rekomendasi bisnis: investigasi transaksi > threshold, kemungkinan bulk purchase atau perlu verifikasi fraud
- Untuk reporting "penjualan harian tipikal", gunakan **median** bukan mean agar tidak inflated oleh transaksi besar